# M3C2 — Outliers, Scaling and Normalization
Open **learning/M3C2_Starter.ipynb** in Colab and save your own copy. Keep the neighbouring **telco_source.csv** ready; upload only that CSV when C01 asks. Follow C00–C21 from top to bottom.
We predict **Churn: Yes / No** — did a customer leave? Inputs describe the customer: months with the company, payments, and later selected service categories. Churn is the answer, never an input. Validation customers already have known Churn answers; we compare each prediction with that recorded answer. Toy example: prediction Yes, known answer No → incorrect; prediction No, known answer No → correct. Accuracy counts correct predictions, not payments.
There are **two empty AI-code cells: C10 and C16**. Ready tools do the repeated measurement; your AI writes the comparison loops. You run and check them.
Code: press ▶. Text note: double-click, replace `...` with your words, then click outside. The mentor shows this once.
If a step fails, stop at that step. Share the exact error and safe evidence with your AI helper or mentor; ask for one small repair. Never remove a check to obtain PASS.

## C00 — Ready tools

Press ▶ once. These prepared tools keep the comparison fair. Their internal code is not today’s writing task.

In [ ]:
#@title C00 — Ready tools
import base64, copy, hashlib, json, zlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

_PINNED = json.loads(zlib.decompress(base64.b64decode('...')))
_SOURCE_SHA = '16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91'
NUMERIC = ['tenure', 'MonthlyCharges', 'TotalCharges']
_ALLOWED = ['Contract', 'PaymentMethod', 'InternetService', 'OnlineSecurity',
            'TechSupport', 'PaperlessBilling', 'Partner', 'Dependents', 'PhoneService']
_records = {'scaling': {}, 'encoding': {}}
selected_fields = ['Contract', 'PaymentMethod', 'InternetService']
scaling_results = None
encoding_results = None
_snapshot = None
_last_saved_token = None

def _assert_source():
    assert hashlib.sha256(Path('telco_source.csv').read_bytes()).hexdigest() == _SOURCE_SHA, 'Source CSV changed.'
    assert raw.equals(_snapshot), 'raw changed. Reload the issued source; keep raw unchanged.'
    assert raw['customerID'].is_unique and not raw.duplicated().any(), 'Unexpected duplicate source.'
    ids = _PINNED['ids']
    parts = [set(ids[k]) for k in ['train','validation','final_test']]
    assert [len(p) for p in parts] == [4225,1409,1409], 'Frozen split count changed.'
    assert not (parts[0]&parts[1] or parts[0]&parts[2] or parts[1]&parts[2]), 'Split overlap.'
    assert set.union(*parts) == set(raw['customerID']), 'Wrong customer set.'
    for key in ['train','validation']:
        expected = _snapshot.set_index('customerID').loc[ids[key]].copy()
        expected['TotalCharges'] = pd.to_numeric(expected['TotalCharges'].astype('string').str.strip(), errors='raise')
        # Blank strings are explicitly unknown, not zero.
        assert frames[key].equals(expected), 'Working rows, order, fields or targets changed: '+key
    assert NUMERIC == ['tenure','MonthlyCharges','TotalCharges'], 'Numeric feature contract changed.'

def load_source():
    global raw, _snapshot, frames, scaling_results, encoding_results
    path = Path('telco_source.csv')
    if not path.exists():
        try:
            from google.colab import files
        except ImportError as e:
            raise FileNotFoundError('Place the issued telco_source.csv beside the notebook.') from e
        print('Upload only the issued telco_source.csv.')
        uploaded = files.upload()
        assert 'telco_source.csv' in uploaded, 'Upload the file with its original name.'
    assert hashlib.sha256(path.read_bytes()).hexdigest() == _SOURCE_SHA, 'Wrong source file. Ask the mentor.'
    raw = pd.read_csv(path)
    _snapshot = raw.copy(deep=True)
    indexed = raw.set_index('customerID')
    frames = {}
    for key in ['train','validation']:
        frame = indexed.loc[_PINNED['ids'][key]].copy()
        frame['TotalCharges'] = pd.to_numeric(frame['TotalCharges'].astype('string').str.strip(), errors='raise')
        frames[key] = frame
    _records['scaling'].clear(); _records['encoding'].clear()
    scaling_results = None; encoding_results = None
    _assert_source()
    missing = frames['train']['TotalCharges'].isna().sum()
    median = frames['train']['TotalCharges'].median()
    assert missing == 8 and median == 1422.1
    display(pd.DataFrame({'Ready fact':['Source customers','Train rows','Validation rows','Final test reserved','Unknown train total payments','Train median total payment'],
                          'Value':[len(raw),4225,1409,1409,int(missing),median]}))
    print('raw is preserved. Train and validation use the frozen M3C1 customer IDs.')
    print('Final test is reserved: no model score or choice uses it.')

def _fields(fields):
    assert isinstance(fields,list) and 2 <= len(fields) <= 3 and len(set(fields)) == len(fields), 'Choose 2–3 different fields.'
    assert all(c in _ALLOWED for c in fields), 'Choose only fields printed in C13; never ID or Churn.'
    return fields.copy()

def _array_token(value):
    arr = np.asarray(value)
    return hashlib.sha256(arr.tobytes()).hexdigest()

def _matrices(kind, method, fields):
    _assert_source()
    train, valid = frames['train'], frames['validation']
    imputer = SimpleImputer(strategy='median').fit(train[NUMERIC])
    a = imputer.transform(train[NUMERIC]); b = imputer.transform(valid[NUMERIC])
    assert imputer.statistics_[2] == 1422.1
    scaler = None
    if kind == 'scaling':
        assert fields == [], 'Scaling compares only the three numeric fields.'
        assert method in ['none','standard','minmax'], 'Unknown scaling method.'
        if method != 'none':
            scaler = (StandardScaler() if method == 'standard' else MinMaxScaler()).fit(a)
            a, b = scaler.transform(a), scaler.transform(b)
        unknown_counts = {}
    else:
        assert kind == 'encoding' and method in ['integer','onehot'], 'Unknown encoding method.'
        fields = _fields(fields)
        # Same three numeric fields, same StandardScaler in BOTH encoding runs.
        scaler = StandardScaler().fit(a)
        a, b = scaler.transform(a), scaler.transform(b)
        if method == 'integer':
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        else:
            encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        ca = encoder.fit_transform(train[fields])
        cb = encoder.transform(valid[fields])
        unknown_counts = {c:int((~valid[c].isin(encoder.categories_[i])).sum()) for i,c in enumerate(fields)}
        a, b = np.column_stack([a,ca]), np.column_stack([b,cb])
    assert a.shape[0] == 4225 and b.shape[0] == 1409 and a.shape[1] == b.shape[1]
    assert np.isfinite(a).all() and np.isfinite(b).all()
    return a,b,unknown_counts

def measure_scaling(method):
    """One ready KNN=5 run. method is 'none', 'standard' or 'minmax'."""
    return _measure('scaling',method,[])

def measure_encoding(method, fields):
    """One ready KNN=5 run. method is 'integer' or 'onehot'; fields is a 2–3 item list."""
    return _measure('encoding',method,fields)

def _measure(kind, method, fields):
    a,b,unknown = _matrices(kind,method,fields)
    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(a, frames['train']['Churn'])
    pred = model.predict(b)
    row = {'method':method,'validation_accuracy':float(accuracy_score(frames['validation']['Churn'],pred)),
           'input_columns':a.shape[1]}
    _records[kind][method] = {'row':row.copy(),'fields':fields.copy(),'train':_array_token(a),'validation':_array_token(b),
                             'prediction':pred.copy(),'unknown':unknown,'ids':copy.deepcopy(_PINNED['ids'])}
    return row

def _validate(kind, results):
    _assert_source()
    methods = ['none','standard','minmax'] if kind == 'scaling' else ['integer','onehot']
    fields = [] if kind == 'scaling' else _fields(selected_fields)
    assert isinstance(results,list) and len(results) == len(methods), 'Unfinished: run the AI comparison cell first.'
    assert [r.get('method') for r in results] == methods, 'Missing, reordered or repeated method.'
    assert set(_records[kind]) == set(methods), 'Each requested method must have a real ready-tool run.'
    for row in results:
        method = row['method']; record = _records[kind][method]
        assert record['fields'] == fields and record['ids'] == _PINNED['ids'], 'Fields or customer IDs changed.'
        a,b,unknown = _matrices(kind,method,fields)
        # Independent audit from the preserved source and frozen IDs, not the run's arrays.
        reference_train = _snapshot.set_index('customerID').loc[_PINNED['ids']['train']].copy()
        reference_valid = _snapshot.set_index('customerID').loc[_PINNED['ids']['validation']].copy()
        for f in [reference_train,reference_valid]:
            f['TotalCharges'] = pd.to_numeric(f['TotalCharges'].astype('string').str.strip(), errors='raise')
        audit_imputer = SimpleImputer(strategy='median').fit(reference_train[['tenure','MonthlyCharges','TotalCharges']])
        audit_a = audit_imputer.transform(reference_train[['tenure','MonthlyCharges','TotalCharges']])
        audit_b = audit_imputer.transform(reference_valid[['tenure','MonthlyCharges','TotalCharges']])
        if kind == 'encoding' or method == 'standard':
            audit_scaler = StandardScaler().fit(audit_a)
            audit_a,audit_b = audit_scaler.transform(audit_a),audit_scaler.transform(audit_b)
        elif method == 'minmax':
            audit_scaler = MinMaxScaler().fit(audit_a)
            audit_a,audit_b = audit_scaler.transform(audit_a),audit_scaler.transform(audit_b)
        if kind == 'encoding':
            audit_encoder = (OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
                             if method == 'integer' else OneHotEncoder(handle_unknown='ignore',sparse_output=False))
            audit_ca = audit_encoder.fit_transform(reference_train[fields])
            audit_cb = audit_encoder.transform(reference_valid[fields])
            audit_a,audit_b = np.column_stack([audit_a,audit_ca]),np.column_stack([audit_b,audit_cb])
        assert a.shape == audit_a.shape and b.shape == audit_b.shape, 'Feature count differs from independent audit.'
        assert np.array_equal(a,audit_a) and np.array_equal(b,audit_b), 'Train-only preprocessing differs from independent audit.'
        assert _array_token(a) == record['train'] and _array_token(b) == record['validation'], 'Feature construction changed.'
        assert record['unknown'] == unknown
        assert row == record['row'], 'Reported table differs from its measured run.'
        score = accuracy_score(frames['validation']['Churn'],record['prediction'])
        assert score == row['validation_accuracy'], 'Reported accuracy is not the observed score.'
        audit_model = KNeighborsClassifier(n_neighbors=5).fit(audit_a, reference_train['Churn'])
        assert np.array_equal(audit_model.predict(audit_b),record['prediction']), 'Predictions differ from fixed KNN=5 audit.'
    return pd.DataFrame(results)

def show_checked(kind):
    results = scaling_results if kind == 'scaling' else encoding_results
    try:
        table = _validate(kind,results)
    except (AssertionError,KeyError,TypeError,ValueError) as e:
        print('NEEDS ATTENTION:',str(e))
        print('Stop HERE. Open the optional repair help immediately below this check; keep all checks. No result is approved.')
        return False
    if kind == 'encoding':
        table = table.copy()
        table['correct_customers'] = [int(np.sum(_records[kind][m]['prediction'] == frames['validation']['Churn'].to_numpy())) for m in ['integer','onehot']]
        table['validation_customers'] = 1409
    display(table)
    print('PASS: original source, fixed customers, train-only fit, same inputs and measured scores checked.')
    print('Final test was not scored. Accuracy here describes only these validation customers.')
    if kind == 'encoding':
        counts = table['correct_customers'].tolist()
        difference = abs(counts[0]-counts[1])
        print(f'Integer: {counts[0]}/1409 correct; one-hot: {counts[1]}/1409 correct. Difference: {difference} customer(s).')
        if difference <= 1:
            print('These scores are almost the same here. Is a difference of at most one customer enough for a general winner? No: it describes this comparison only.')
        else:
            print('This difference describes your current fields and these validation customers only. It does not establish a universal winner.')
        print('Changing fields can change the counts. Your prediction may differ from the result; that is not a failed experiment.')
        display(pd.DataFrame([_records[kind][m]['unknown'] for m in ['integer','onehot']], index=['integer','onehot']).rename_axis('Unknown validation category counts'))
    return True

def make_request(kind):
    _assert_source()
    common = """I am a beginner using Google Colab. Give ONE short Python code cell, not a claim of execution. Return only Python, without Markdown fences or explanation.
Context: pandas is already imported as pd. The issued Telco source and the exact M3C1 ID map
are loaded. Train=4225 customers; validation=1409; final test=1409 is reserved, never scored.
Each row is one customer. Churn is the answer; customerID is not a model input.
The ready tools keep rows and answers fixed, fit the missing-value rule on train only
(TotalCharges median 1422.1, 8 unknown train values), and transform validation.
Numeric inputs: tenure (months), MonthlyCharges (one month's payment), TotalCharges (total recorded payment).
KNN predicts from the five nearest training customers; the ready tool fixes KNeighborsClassifier(n_neighbors=5).
Accuracy is the share of correct validation predictions, e.g. 7 correct / 10 = 70%.
Do not change raw, frames, customer IDs, NUMERIC, ready tools, checks or model settings.
Do not delete outliers, add inputs, tune parameters, access final-test rows, write files or invent results.
Use only the limited ready interface below; do not redefine functions or inspect private variables.
I run the returned code in Colab and read the checks. Attach no CSV or private customer rows.
This WHOLE request includes Context; no earlier table or helper source needs to be copied.
"""
    if kind == 'scaling':
        task = """Task: compare three scaling methods on exactly the three numeric inputs.
Ready interface: measure_scaling(method) returns one dictionary with keys method,
validation_accuracy and input_columns. It runs the fixed model once, with train-only
median filling and the requested train-fitted scaler. Allowed method strings:
'none' (no scaling), 'standard' (StandardScaler), 'minmax' (MinMaxScaler).
Write a short loop calling this tool once for each method in that order.
Collect the returned dictionaries in a list named scaling_results.
Display pd.DataFrame(scaling_results). No other operations are needed."""
    else:
        fields = _fields(selected_fields)
        safe = {c:sorted(frames['train'][c].dropna().unique().tolist()) for c in fields}
        task = f"""Task: compare integer codes and one-hot on the SAME selected fields.
Current selected_fields variable: {fields!r}
Actual train categories (safe words only): {json.dumps(safe,ensure_ascii=False)}
Integer numbers label categories but do not prove a ranking. One-hot makes a separate
0/1 input for each train category. Train-fitted integer mapping uses -1 for an unseen
category; one-hot uses all zeros for that field. Unknown counts are checked separately.
Both runs use the same numeric inputs with the SAME train-fitted StandardScaler.
Ready interface: measure_encoding(method, fields) returns a dictionary with keys
method, validation_accuracy and input_columns. It fits the encoder on train only and
runs fixed KNN=5. Allowed method strings: 'integer', 'onehot'.
Write a short loop calling this tool once for each method in that order, passing
selected_fields as the second argument each time. Collect the returned dictionaries
in a list named encoding_results. Display pd.DataFrame(encoding_results).
Do not change selected_fields inside this code. No other operations are needed."""
    print(common+'\n'+task)

def repair_request(kind):
    """Print a fresh safe repair packet; never print customer rows or IDs."""
    make_request(kind)
    slot, check = ('C10','C11') if kind == 'scaling' else ('C16','C17')
    print('\nREPAIR TASK: The comparison above failed in my own Colab. Repair only my comparison loop; preserve every check and condition.')
    print('Expected: each allowed method runs once, then the checked measured table passes. Actual result: [paste exact error / NEEDS ATTENTION line here].')
    print('CURRENT CODE: [paste the entire current '+slot+' code here, including a wrong method name or accidental fences].')
    print('Return only Python without Markdown fences, explanation, function redefinitions or extra data access. I will replace '+slot+', run it myself, then rerun '+check+'. Do not claim you ran Colab.')
    print('Copy this WHOLE output plus the two replacements above. Include no CSV, customer rows, IDs, private paths or private values. If a traceback contains any, copy only its safe error line.')

def final_evidence():
    print('Fresh evidence — scaling:')
    one = show_checked('scaling')
    print('Fresh evidence — encoding:')
    two = show_checked('encoding')
    print('Current selected categorical fields:',selected_fields)
    return one and two

def save_results():
    global _last_saved_token
    try:
        a = _validate('scaling',scaling_results); b = _validate('encoding',encoding_results)
    except (AssertionError,KeyError,TypeError,ValueError) as e:
        print('NOT SAVED:',str(e))
        print('Any older results CSV is stale; it does not prove the current work passed.')
        return False
    out = pd.concat([a.assign(experiment='scaling'),b.assign(experiment='encoding')],ignore_index=True)
    out.to_csv('M3C2_results.csv',index=False)
    reopened = pd.read_csv('M3C2_results.csv')
    assert reopened.shape == out.shape and np.allclose(reopened.validation_accuracy,out.validation_accuracy)
    _last_saved_token = hashlib.sha256(Path('M3C2_results.csv').read_bytes()).hexdigest()
    print('Optional results CSV saved and reopened. Save your notebook with outputs and your own paragraphs.')
    return True

print('Ready. Next: C01. The tools measure; your AI writes the two comparison loops.')

## C01 — Open the same source

In M3C1 we inspected and cleaned data while preserving the source. We open that preserved source again: this ready cell repeats the checked preparation in working copies with the same customer groups. Today we change how inputs are represented. Press ▶. Read the short facts below. `raw` stays unchanged. Missing total payments use the **train** median only. This lesson reuses the exact M3C1 customer groups; you do not create a new split. The final test stays reserved.

In [ ]:
#@title C01 — Load and check the issued source
load_source()

## C02 — A small unusual value

Predict: which value needs a closer look? The middle half of this list lies from **12 to 16**. Its width is **IQR = 4**. A common rule marks values below **6** or above **22**. A flag means “inspect”, not “delete”. This is a toy list, not Telco.

In [ ]:
#@title C02 — Small outlier example
toy = pd.Series([10,11,12,13,14,15,16,17,100])
q1,q3 = toy.quantile([.25,.75]); iqr = q3-q1
low,high = q1-1.5*iqr,q3+1.5*iqr
display(pd.DataFrame({'value':toy,'IQR flag':(toy<low)|(toy>high)}))
print(f'Q1={q1:g}, Q3={q3:g}, IQR={iqr:g}; bounds {low:g} to {high:g}')
z = (toy-toy.mean())/toy.std(ddof=0)
print('Second signal, z-score: distance from the mean in typical-spread units.')
print('Toy values with |z| > 3:', toy[z.abs()>3].tolist())
print('These two signals can disagree. Neither proves a data error.')

## C03 — Inspect real train payments

Press ▶. The box marks the middle half; the histogram shows how values are spread. We inspect train only. A large payment can be real. Read the rule’s count before calling anything an outlier.

In [ ]:
#@title C03 — Real train inspection, no deletion
_assert_source()
train_numeric = frames['train'][NUMERIC]
q = train_numeric.quantile([.25,.75]); spread = q.loc[.75]-q.loc[.25]
lower, upper = q.loc[.25]-1.5*spread, q.loc[.75]+1.5*spread
flags = train_numeric.lt(lower) | train_numeric.gt(upper)
z = (train_numeric-train_numeric.mean())/train_numeric.std(ddof=0)
display(pd.DataFrame({'Q1':q.loc[.25], 'Q3':q.loc[.75], 'lower bound':lower,'upper bound':upper,
                      'IQR flagged':flags.sum(),'|z| > 3':z.abs().gt(3).sum()}))
fig, axes = plt.subplots(2,3,figsize=(11,5))
for i,c in enumerate(NUMERIC):
    axes[0,i].boxplot(train_numeric[c].dropna()); axes[0,i].set_title(c)
    axes[1,i].hist(train_numeric[c].dropna(),bins=20); axes[1,i].set_xlabel(c)
fig.tight_layout(); display(fig); plt.close(fig)
candidate = train_numeric.loc[[train_numeric['TotalCharges'].idxmax()]].reset_index(drop=True)
display(candidate)
print('Candidate above: highest known train total payment, shown without customer ID.')
print('It is a case to inspect, NOT an IQR-flagged outlier. No customer is removed.')

## C04 — Make an evidence-based decision

Write immediately from the output above.

The real IQR check flagged ... values. The high-payment candidate has ... months and ... total payment.
I would keep / investigate / remove this case because ... . Evidence still needed: ... .
Why a flag alone cannot prove an error: ... .

For today’s fair comparison, keep every customer. Zero flags is a valid finding; do not invent an outlier.

## C05 — Why units can change neighbours

KNN asks the five nearest training customers for a prediction. “Nearest” depends on distances between input numbers. Predict: could the larger unit dominate? These are **squared distances** from (0, 0), not Telco model scores.

In [ ]:
#@title C05 — Small distance example
display(pd.DataFrame({'toy point':['(1, 100)','(10, 0)'], 'squared distance':[1**2+100**2,10**2+0**2]}))
print('100 in the second field dominates 1 in the first. Changing units can change who is nearest.')

## C06 — Two ways to put numbers on a comparable scale

MinMax learns train’s smallest and largest values. Standard subtracts train’s average and divides by train’s typical spread. Predict what happens to a **new 50** when train was 10, 20, 30. Watch the prepared example.

In [ ]:
#@title C06 — Learn on train, transform new values
tiny_train = np.array([[10.],[20.],[30.]])
tiny_new = np.array([[50.]])
mm = MinMaxScaler().fit(tiny_train); std = StandardScaler().fit(tiny_train)
display(pd.DataFrame({'train original':tiny_train.ravel(),'MinMax':mm.transform(tiny_train).ravel(),
                      'Standard':std.transform(tiny_train).ravel()}))
print('New 50 after MinMax:',mm.transform(tiny_new).item())
print('New 50 after Standard:',round(std.transform(tiny_new).item(),3))
print('New MinMax values may leave 0..1. Standard may be negative; it does not make a bell-shaped distribution.')

## C07 — Keep the comparison fair

We change **one thing: scaling**. Three runs: no scaling → Standard → MinMax.

The ready measurement keeps the same train/validation customers, three numeric inputs, train-only median filling and **KNN with five neighbours**. `customerID` and `Churn` never enter the inputs. Only train teaches the missing-value rule and scaler; validation receives those rules.

**Accuracy** = correct answers / all validation answers. Seven correct out of ten means **70%**. A higher validation score here is evidence for this comparison, not a promise for all new customers. Do not use final test to choose.

## C08 — Predict and ask in your own words

My prediction: ... will have the highest validation accuracy because ... .

My request to AI, before using the support template: ... .

Describe the change, what must stay fixed, and how you will check it. A prediction can be wrong; the measured table decides what happened.

## C09 — Get one complete scaling request

Press ▶. Copy the **WHOLE output including Context** to your familiar AI helper. No earlier tables or helper code are needed; attach no CSV. Read its proposed loop before pasting only the Python code below.

In [ ]:
#@title C09 — Complete scaling request
make_request('scaling')

## C10 — Run your AI scaling code

Paste the short comparison code returned for the request immediately above. Press ▶ yourself. This is the first empty AI-code slot.

In [ ]:
# Paste the AI scaling comparison here.

## C11 — Check and read a fresh scaling table

Press ▶. Read the table immediately below. If a check fails, stop here and open the optional repair help immediately below. Replace C10 with repaired Python, run C10, then run this C11 check again. Keep the checks.

In [ ]:
#@title C11 — Check current scaling results
scaling_ok = show_checked('scaling')

### Optional repair help — C11

Open only if this step fails. If it passed, continue to the next numbered section.
1. If the reply contains ```python or ```, keep only the Python lines inside; replace the whole C10 code, run C10, then run C11 again.
2. For another error, run the safe repair-packet cell below. Copy its WHOLE output to your AI helper. Replace its CURRENT CODE placeholder with all code currently in C10; replace Actual result with the exact safe error line from your failed run. The complete current Context and permitted API are printed here again: do not search above or send CSV, customer rows or IDs.
3. Replace the entire original C10 code with the repaired Python only. Run C10 yourself, then rerun C11. A chat reply is not a checked result. Keep the checks; if it still fails, use the new error and current code in a new packet.
4. Runtime reset / NameError / C00 or C01 rerun? Your old printed tables can be stale. Keep your written notes, run C00 and C01 (upload the issued CSV again if asked), then follow the numbered route through C09 → C10 → C11 and C14 → C15 → C16 → C17 → C19 → C21. Rerun both AI slots with the current prompts; earlier output is not current evidence.

In [ ]:
#@title Optional — fresh safe repair packet
repair_request('scaling')

## C12 — Write the scaling paragraph now

Use the table immediately above; write **one paragraph** in your own words.

I compared ... while keeping ... fixed. My validation accuracies were ... . My prediction ... . Scaling changed / did not change the result because ... . This supports ... for these validation customers, but does not prove ... .

If two scores tie, state the tie. Do not invent an improvement.

## C13 — Words need numbers, but numbers can suggest a false order

“Month-to-month”, “One year”, and “Two year” are Contract categories. Integer codes label words; they can create artificial distances. One-hot adds a separate yes/no column for each word. Read the actual train words below before choosing fields.

In [ ]:
#@title C13 — Actual safe category words and a toy encoding
_assert_source()
display(pd.DataFrame([{'field':c,'actual train categories':', '.join(sorted(frames['train'][c].dropna().unique()))} for c in _ALLOWED]))
display(pd.DataFrame({'toy colour':['red','green','blue'],'integer label':[0,1,2],
                      'is red':[1,0,0],'is green':[0,1,0],'is blue':[0,0,1]}))
print('Toy colours have no size ranking. Integer label 2 is not twice the colour of label 1.')
print('Unseen words: integer uses -1; one-hot uses all zeros for that field. Count unknown words separately.')

## C14 — Choose 2–3 fields

This small cell is **your input choice**, not an AI-code task. Keep or edit the list using the real field names immediately above, then press ▶. Both encoding methods must use this same list.

My reason for these fields: ... . My prediction about integer versus one-hot: ... . My own request to AI: ... .

In [ ]:
#@title C14 — Your selected categorical inputs
selected_fields = ['Contract', 'PaymentMethod', 'InternetService']
_fields(selected_fields)
print('Both runs will use:',selected_fields)

## C15 — Get one complete encoding request

Press ▶ after choosing fields. Copy the **WHOLE output including Context**. It contains your current fields and their actual train words. No earlier table or CSV attachment is needed. Read the reply; both runs must keep the same numeric Standard scaling.

In [ ]:
#@title C15 — Complete encoding request
make_request('encoding')

## C16 — Run your AI encoding code

Paste only the Python code returned for the complete request immediately above. Press ▶. This is the second empty AI-code slot.

In [ ]:
# Paste the AI encoding comparison here.

## C17 — Check and read a fresh encoding table

Press ▶. Read current scores and unknown counts. If anything fails, open the optional repair help immediately below. Replace C16 with repaired Python, run C16, then this C17 check again. After changing C14, rerun C15 for its current prompt and BOTH methods in C16. Do not mix old and new fields.

In [ ]:
#@title C17 — Check current encoding results
encoding_ok = show_checked('encoding')

### Optional repair help — C17

Open only if this step fails. If it passed, continue to the next numbered section.
1. If the reply contains ```python or ```, keep only the Python lines inside; replace the whole C16 code, run C16, then run C17 again.
2. For another error, run the safe repair-packet cell below. Copy its WHOLE output to your AI helper. Replace its CURRENT CODE placeholder with all code currently in C16; replace Actual result with the exact safe error line from your failed run. The complete current Context and permitted API are printed here again: do not search above or send CSV, customer rows or IDs.
3. Replace the entire original C16 code with the repaired Python only. Run C16 yourself, then rerun C17. A chat reply is not a checked result. Keep the checks; if it still fails, use the new error and current code in a new packet.
4. Changed C14 fields? Run C15 again for the current prompt, then run BOTH methods in C16 again and C17. Reusing one old method is not a new comparison.
5. Runtime reset / NameError / C00 or C01 rerun? Your old printed tables can be stale. Keep your written notes, run C00 and C01 (upload the issued CSV again if asked), then follow the numbered route through C09 → C10 → C11 and C14 → C15 → C16 → C17 → C19 → C21. Rerun both AI slots with the current prompts; earlier output is not current evidence.

In [ ]:
#@title Optional — fresh safe repair packet
repair_request('encoding')

## C18 — Write the encoding paragraph now

Use the table immediately above; write **one paragraph**.

I chose ... because ... . Integer codes mean ... ; one-hot means ... . The accuracies were ... with ... input columns. My prediction ... . The comparison supports ... for these customers; it does not prove ... . Unknown categories were ... and handled by ... .

These encoding scores also use categorical inputs. Do not attribute a difference from the earlier numeric-only table solely to encoding.

## C19 — Refresh the evidence for your explanation

Press ▶. The required facts appear here again so you can explain your work without searching earlier tables. Continue only after both checks pass.

In [ ]:
#@title C19 — Fresh final evidence
all_ok = final_evidence()

## C20 — Explain to a partner

Use the fresh evidence immediately above. Your partner asks a question; they do not write your assessed answer. Working alone? Write one question and answer it yourself using C19.

Transfer question: We express the same monthly payment in smaller currency units. Did the customer change? Why might the distance used by the model change? Explain in your own words.

PASS checks the measured tables; it does not check your understanding or fill your paragraphs.

My partner asked: ... . I explained: ... .

What stayed fixed in each comparison: ... . Why learning rules from validation would be unfair: ... .
My AI use (unavailable / proposed code / I ran and checked it): ... .

Your required submission contains the **executed notebook**, the **five measured accuracies**, **one scaling paragraph** and **one encoding paragraph**. Follow the announced LMS deadline. The mentor reference is not your own successful run.

## C21 — Save your own executed notebook

Press ▶ to recheck current results. The CSV is an optional convenience, not an extra submission requirement. In Colab use **File → Download → Download .ipynb** to keep outputs and your notes. Reopen the saved notebook and confirm that both tables and your two paragraphs are present. Keep the source CSV unchanged for the next lesson.

In [ ]:
#@title C21 — Guarded optional results export
saved = save_results()